# Bölüm 13 — BÖLÜM 13: VERİ AKIŞI İŞLEME VE GERÇEK ZAMANLI ANALİTİK

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 13. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q apache-flink kafka-python nltk pyspark river transformers tweepy


## 13.2. Apache Kafka: Dağıtık Mesajlaşma ve Veri Akışı Platformu


### 13.2.3. Python ile Uygulama: Kafka Üretici ve Tüketici Ağı

`bolum13/13_02_03_python-ile-uygulama-kafka-uretici-ve-tuketici-ag.py`

_Kitap: Kod 13.1, Kod 13.2_


In [ ]:
from kafka import KafkaProducer
from kafka.errors import KafkaError
import json
import time
import random
import uuid
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ---- Kafka Producer Konfigürasyonu ----
def create_producer(bootstrap_servers: list) -> KafkaProducer:
    """
    Kafka Producer oluşturur. Parametre açıklamaları:
    - value_serializer: Python dict'i JSON byte'ına çevirir (serileştirme)
    - key_serializer:   Partition anahtarını UTF-8 byte'ına çevirir
    - acks='all':       Tüm in-sync replica'lar yazıyı onaylayana kadar bekle (güvenilir)
    - retries:          Başarısız yazımda 3 kez yeniden dene
    - compression_type: lz4 sıkıştırma — yüksek hacimde ağ ve depolama tasarrufu
    - linger_ms:        Mesajları 5ms biriktir, sonra toplu gönder (throughput artışı)
    """
    return KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8'),
        key_serializer=lambda k: k.encode('utf-8') if k else None,
        acks='all',
        retries=3,
        compression_type='lz4',
        linger_ms=5,
        batch_size=65536  # 64KB — toplu gönderim için tampon boyutu
    )

def on_send_success(record_metadata):
    logger.info(f"Mesaj gönderildi | Topic: {record_metadata.topic} | "
                f"Partition: {record_metadata.partition} | Offset: {record_metadata.offset}")

def on_send_error(excp):
    logger.error(f"Mesaj gönderim hatası: {excp}")

# ---- Ana Üretici Döngüsü ----
def run_producer():
    producer = create_producer(['localhost:9092'])
    TOPIC = 'user_clickstream'
    PRODUCTS = ['laptop', 'phone', 'headphones', 'tablet', 'smartwatch', 'camera']
    CATEGORIES = ['elektronik', 'giyim', 'ev-bahce', 'spor', 'kitap']
    ACTIONS = ['view_product', 'add_to_cart', 'remove_from_cart', 'checkout', 'purchase', 'search']

    logger.info(f"Producer başlatıldı. Topic: {TOPIC}")
    sent_count = 0

    try:
        while True:
            user_id = f"user_{random.randint(1000, 9999)}"

            # Her mesaj için partition key olarak user_id kullanılır.
            # Aynı kullanıcının tüm olayları aynı partition'a gider => sıra garantisi
            event = {
                'event_id':    str(uuid.uuid4()),
                'user_id':     user_id,
                'session_id':  f"session_{random.randint(100, 999)}",
                'action':      random.choice(ACTIONS),
                'product':     random.choice(PRODUCTS),
                'category':    random.choice(CATEGORIES),
                'price':       round(random.uniform(9.99, 4999.99), 2),
                'quantity':    random.randint(1, 5),
                'device':      random.choice(['mobile', 'desktop', 'tablet']),
                'country':     random.choice(['TR', 'DE', 'US', 'GB', 'FR']),
                'timestamp':   time.time(),
                'event_time':  time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            }

            # Asenkron gönderim (callback ile) — throughput için kritik
            future = producer.send(
                TOPIC,
                key=user_id,    # Aynı kullanıcı => aynı partition
                value=event
            )
            future.add_callback(on_send_success).add_errback(on_send_error)

            sent_count += 1
            if sent_count % 100 == 0:
                producer.flush()  # Arabellekteki mesajları zorla gönder
                logger.info(f"Toplam gönderilen: {sent_count} mesaj")

            time.sleep(0.1)  # 10 mesaj/saniye hızında üretim

    except KeyboardInterrupt:
        logger.info("Producer durduruluyor...")
    finally:
        producer.flush()
        producer.close()
        logger.info(f"Producer kapatıldı. Toplam gönderilen: {sent_count} mesaj")

if __name__ == '__main__':
    run_producer()

# ============================================================
# KAFKA CONSUMER (TÜKETİCİ) — Gerçek Zamanlı Analitik Motoru
# ============================================================
from kafka import KafkaConsumer
from kafka.errors import KafkaError
import json
import logging
from collections import defaultdict, deque
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ---- Kayan Pencere Tabanlı İstatistik Takipçisi ----
class SlidingWindowTracker:
    """
    Her kullanıcı için son N olaydaki harcama toplamını
    O(1) bellek ve O(1) güncelleme karmaşıklığıyla takip eder.
    Bu yapı, akan veri üzerinde Konsept Kaymasına duyarlı anomali tespiti için temeldir.
    """
    def __init__(self, window_size: int = 10):
        self.window_size = window_size
        self.user_events = defaultdict(lambda: deque(maxlen=window_size))
        self.user_totals = defaultdict(float)

    def add_event(self, user_id: str, price: float, action: str) -> dict:
        window = self.user_events[user_id]

        # Pencere dolduysa en eski değeri toplamdan çıkar
        if len(window) == self.window_size:
            old_price, old_action = window[0]
            if old_action in ('purchase', 'checkout'):
                self.user_totals[user_id] -= old_price

        window.append((price, action))
        if action in ('purchase', 'checkout'):
            self.user_totals[user_id] += price

        return {
            'user_id':         user_id,
            'window_total':    round(self.user_totals[user_id], 2),
            'event_count':     len(window),
            'avg_price':       round(self.user_totals[user_id] / max(len(window), 1), 2)
        }

# ---- Kafka Consumer Konfigürasyonu ----
def create_consumer(topic: str, group_id: str, bootstrap_servers: list) -> KafkaConsumer:
    """
    Consumer parametreleri:
    - auto_offset_reset='latest':  Yalnızca bu consumer başladıktan sonra gelen
                                    mesajları işle (canlı akış modu)
    - enable_auto_commit=False:    Offset'i manuel yönetiyoruz (exactly-once için)
    - max_poll_records=100:        Tek poll çağrısında en fazla 100 mesaj al
    - session_timeout_ms=30000:    Consumer 30sn cevap vermezse ölü kabul edilir
    """
    return KafkaConsumer(
        topic,
        bootstrap_servers=bootstrap_servers,
        group_id=group_id,
        auto_offset_reset='latest',
        enable_auto_commit=False,  # Manuel commit — exactly-once için
        value_deserializer=lambda x: json.loads(x.decode('utf-8')),
        key_deserializer=lambda x: x.decode('utf-8') if x else None,
        max_poll_records=100,
        session_timeout_ms=30000,
        heartbeat_interval_ms=10000
    )

# ---- Anomali Tespit Kuralları ----
ANOMALY_RULES = {
    'yuksek_degerli_islem':   lambda e: e['action'] == 'purchase' and e['price'] > 2000,
    'hizli_sepet_dolumu':     lambda e: e['action'] == 'add_to_cart' and e['quantity'] > 4,
    'farkli_ulke_aniden':     lambda e: e.get('country') not in ['TR', 'DE']
}

def check_anomalies(event: dict, tracker_stats: dict) -> list:
    anomalies = []
    for rule_name, rule_fn in ANOMALY_RULES.items():
        try:
            if rule_fn(event):
                anomalies.append(rule_name)
        except Exception:
            pass

    # Pencere toplamına dayalı ek kural
    if tracker_stats['window_total'] > 5000:
        anomalies.append('pencere_harcama_limiti_asimi')

    return anomalies

# ---- Ana Tüketici Döngüsü ----
def run_consumer():
    consumer = create_consumer('user_clickstream', 'ml_analytics_group', ['localhost:9092'])
    tracker = SlidingWindowTracker(window_size=10)
    processed = 0

    logger.info("Consumer başlatıldı. Mesajlar dinleniyor...")

    try:
        for message in consumer:
            event = message.value
            user_id = message.key

            # 1. Kayan Pencere İstatistiklerini Güncelle
            stats = tracker.add_event(user_id, event['price'], event['action'])

            # 2. Anomali Tespiti
            anomalies = check_anomalies(event, stats)

            if anomalies:
                logger.warning(
                    f"ANOMALI TESPIT | Kullanıcı: {user_id} | "
                    f"Kurallar: {anomalies} | Pencere Toplam: {stats['window_total']} TL"
                )
                # Gerçek sistemde: fraud alert göndermek, hesabı askıya almak vb.

            # 3. Offset'i Manuel Commit Et (exactly-once için)
            # Sadece başarılı işlemden sonra commit yapılır
            consumer.commit({
                message.topic_partition: message.offset + 1
            } if hasattr(message, 'topic_partition') else None)
            # Basit versiyon:
            consumer.commit()

            processed += 1
            if processed % 1000 == 0:
                logger.info(f"İşlenen toplam mesaj: {processed}")

    except KeyboardInterrupt:
        logger.info("Consumer durduruluyor...")
    finally:
        consumer.close()

if __name__ == '__main__':
    run_consumer()

# ============================================================
# KAFKA ADMIN — Topic Oluşturma ve Yapılandırma
# ============================================================
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

def create_kafka_topics(bootstrap_servers: list, topics_config: list):
    """
    topics_config: [{'name': str, 'partitions': int, 'replication_factor': int, 'config': dict}]
    """
    admin = KafkaAdminClient(bootstrap_servers=bootstrap_servers, client_id='admin-client')

    new_topics = [
        NewTopic(
            name=t['name'],
            num_partitions=t['partitions'],
            replication_factor=t['replication_factor'],
            topic_configs=t.get('config', {})
        )
        for t in topics_config
    ]

    try:
        result = admin.create_topics(new_topics=new_topics, validate_only=False)
        for topic, error in result.topic_errors:
            if error is None or error == 0:
                print(f"Topic '{topic}' başarıyla oluşturuldu.")
            else:
                print(f"Topic '{topic}' oluşturma hatası: {error}")
    except TopicAlreadyExistsError:
        print("Topic zaten mevcut.")
    finally:
        admin.close()

# Kullanım Örneği:
create_kafka_topics(
    bootstrap_servers=['localhost:9092'],
    topics_config=[
        {
            'name': 'user_clickstream',
            'partitions': 12,            # 12 paralel okuma kapasitesi
            'replication_factor': 3,     # 3 kopya — 2 broker arızasına dayanıklı
            'config': {
                'retention.ms':      str(7 * 24 * 60 * 60 * 1000),  # 7 gün saklama
                'cleanup.policy':    'delete',   # Retention süresi dolunca sil
                'compression.type':  'lz4',      # Depolama sıkıştırması
                'min.insync.replicas': '2'       # En az 2 replica sync olmalı
            }
        },
        {
            'name': 'fraud_alerts',
            'partitions': 3,
            'replication_factor': 3,
            'config': {
                'retention.ms':    str(30 * 24 * 60 * 60 * 1000),  # 30 gün saklama
                'cleanup.policy':  'compact,delete'  # Compaction + retention
            }
        }
    ]
)


## 13.3. Apache Flink ile Gerçek Zamanlı Veri İşleme Motoru


### 13.3.3. Veri Akışı Operatörleri: Matematiksel Temelleri ve Python Uygulamaları

`bolum13/13_03_03_veri-akisi-operatorleri-matematiksel-temelleri-v.py`

_Kitap: Kod 13.3_


In [ ]:
from pyflink.datastream import StreamExecutionEnvironment, TimeCharacteristic
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer, FlinkKafkaProducer
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
from pyflink.datastream.functions import (
    MapFunction, FilterFunction, FlatMapFunction,
    ReduceFunction, ProcessWindowFunction, KeyedProcessFunction
)
from pyflink.datastream.window import TumblingEventTimeWindows, SlidingEventTimeWindows
from pyflink.common.time import Time
from pyflink.datastream.state import ValueStateDescriptor
import json
import logging

logging.basicConfig(level=logging.INFO)

# ---- 1. MAP: JSON dönüşümü + özellik çıkarımı ----
class EventParserMap(MapFunction):
    """Kafka'dan gelen ham JSON string'i Python dict'ine çevirir."""
    def map(self, value):
        try:
            event = json.loads(value)
            # Türetilmiş özellik: toplam değer
            event['total_value'] = event.get('price', 0) * event.get('quantity', 1)
            return json.dumps(event)
        except json.JSONDecodeError:
            return None  # Bozuk veri filtre edilecek

# ---- 2. FILTER: Bozuk ve gereksiz olayları elee ----
class ValidEventFilter(FilterFunction):
    """Yalnızca geçerli ve anlamlı olayları akışta bırakır."""
    VALID_ACTIONS = {'purchase', 'add_to_cart', 'checkout', 'view_product'}

    def filter(self, value):
        if value is None:
            return False
        try:
            event = json.loads(value)
            return (
                event.get('action') in self.VALID_ACTIONS and
                event.get('price', 0) > 0 and
                event.get('user_id') is not None
            )
        except Exception:
            return False

# ---- 3. FLATMAP: Satın alma olayından birden fazla analiz üret ----
class PurchaseEventExpander(FlatMapFunction):
    """
    Her 'purchase' olayı için hem global hem de kullanıcı bazlı
    analiz mesajları üretir (bire-çok dönüşüm).
    """
    def flat_map(self, value):
        event = json.loads(value)
        if event.get('action') != 'purchase':
            yield value  # Diğer olaylar değişmeden geçsin
            return
        # Orijinal olay
        yield value
        # Kullanıcı segmentasyon mesajı
        segment_msg = {
            'type': 'user_segment_update',
            'user_id': event['user_id'],
            'segment': 'high_value' if event['total_value'] > 1000 else 'standard',
            'timestamp': event['timestamp']
        }
        yield json.dumps(segment_msg)

# ---- 4. REDUCE: Pencere içinde toplam harcama birikim ----
class RevenueReducer(ReduceFunction):
    """
    İki olay arasında toplam geliri biriktirir.
    S_t = f_reduce(S_{t-1}, x_t) kalıbını uygular.
    """
    def reduce(self, value1, value2):
        ev1 = json.loads(value1)
        ev2 = json.loads(value2)
        ev1['total_value'] = ev1.get('total_value', 0) + ev2.get('total_value', 0)
        ev1['event_count']  = ev1.get('event_count', 1) + 1
        return json.dumps(ev1)

# ---- 5. KEYED PROCESS FUNCTION: Fraud tespiti için stateful işlem ----
class FraudDetector(KeyedProcessFunction):
    """
    Kullanıcı başına son N işlemin toplamını takip eder.
    Belirli eşiği geçen kullanıcılar için fraud uyarısı üretir.
    """
    FRAUD_THRESHOLD = 5000.0  # 5000 birim üstü şüpheli
    WINDOW_SECONDS  = 300     # 5 dakikalık pencere

    def open(self, runtime_context):
        # Kullanıcı bazında birikimli harcama state'i
        self.total_spend = runtime_context.get_state(
            ValueStateDescriptor('user_spend', Types.DOUBLE())
        )
        # Son olay zamanı (pencere sıfırlama için)
        self.last_event_time = runtime_context.get_state(
            ValueStateDescriptor('last_event_time', Types.LONG())
        )

    def process_element(self, value, ctx):
        event = json.loads(value)
        current_time = ctx.timestamp() or int(event.get('timestamp', 0) * 1000)

        # Eski state'i al
        current_spend = self.total_spend.value() or 0.0
        last_time = self.last_event_time.value() or current_time

        # Pencere süresi dolduysa state'i sıfırla
        if (current_time - last_time) > self.WINDOW_SECONDS * 1000:
            current_spend = 0.0

        # State güncelle
        new_spend = current_spend + event.get('total_value', 0)
        self.total_spend.update(new_spend)
        self.last_event_time.update(current_time)

        # Fraud kontrol
        if new_spend > self.FRAUD_THRESHOLD:
            alert = {
                'alert_type':   'POTENTIAL_FRAUD',
                'user_id':      event['user_id'],
                'window_spend': new_spend,
                'threshold':    self.FRAUD_THRESHOLD,
                'timestamp':    event['timestamp']
            }
            yield json.dumps(alert)
        else:
            yield value

# ---- MAIN: Pipeline Tanımı ----
def build_pipeline():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_stream_time_characteristic(TimeCharacteristic.EventTime)
    env.set_parallelism(4)  # 4 paralel görev

    # Checkpoint: exactly-once garantisi için
    env.enable_checkpointing(30000)  # Her 30sn checkpoint

    # Kafka Source
    kafka_props = {'bootstrap.servers': 'localhost:9092', 'group.id': 'flink_pipeline'}
    kafka_source = FlinkKafkaConsumer(
        topics='user_clickstream',
        deserialization_schema=SimpleStringSchema(),
        properties=kafka_props
    )
    kafka_source.set_start_from_latest()

    # Pipeline Zinciri
    stream = env.add_source(kafka_source)              \
        .map(EventParserMap(), output_type=Types.STRING())    \
        .filter(ValidEventFilter())                    \
        .flat_map(PurchaseEventExpander(), output_type=Types.STRING())

    # Key bazlı fraud tespiti (stateful)
    fraud_stream = stream \
        .key_by(lambda x: json.loads(x).get('user_id', 'unknown')) \
        .process(FraudDetector(), output_type=Types.STRING())

    # Tumbling Window ile 5 dakikalık gelir toplamı
    revenue_stream = stream \
        .filter(lambda x: json.loads(x).get('action') == 'purchase') \
        .key_by(lambda x: json.loads(x).get('category', 'unknown')) \
        .window(TumblingEventTimeWindows.of(Time.minutes(5))) \
        .reduce(RevenueReducer())

    # Sonuçları yazdır (gerçekte başka bir Kafka topic'e veya DB'e yazılır)
    fraud_stream.print()
    revenue_stream.print()

    env.execute('Ecommerce_Realtime_Analytics_Job')

if __name__ == '__main__':
    build_pipeline()

# ================================================================
# Kayan Pencere (Sliding Window) ile Fiyat Anomali Tespiti
# ================================================================
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.window import SlidingEventTimeWindows
from pyflink.datastream.functions import AggregateFunction, ProcessWindowFunction
from pyflink.common.time import Time
from pyflink.common.typeinfo import Types
import json, math

class PriceStatsAggregator(AggregateFunction):
    """
    Kayan penceredeki fiyat istatistiklerini online olarak toplar.
    Welford'un online varyans algoritmasını kullanır:
    M_k = M_{k-1} + (x_k - M_{k-1}) / k
    S_k = S_{k-1} + (x_k - M_{k-1}) * (x_k - M_k)
    Varyans = S_k / (k - 1)  [örneklem varyansı]
    """
    def create_accumulator(self):
        # (count, mean, M2, min, max)
        return (0, 0.0, 0.0, float('inf'), float('-inf'))

    def add(self, value, accumulator):
        event = json.loads(value)
        price = event.get('price', 0)
        count, mean, M2, mn, mx = accumulator
        count += 1
        delta  = price - mean
        mean  += delta / count
        delta2 = price - mean
        M2    += delta * delta2
        return (count, mean, M2, min(mn, price), max(mx, price))

    def get_result(self, accumulator):
        count, mean, M2, mn, mx = accumulator
        variance = M2 / (count - 1) if count > 1 else 0.0
        std_dev  = math.sqrt(variance)
        return json.dumps({
            'count':    count,
            'mean':     round(mean, 2),
            'std_dev':  round(std_dev, 2),
            'min':      mn,
            'max':      mx
        })

    def merge(self, acc1, acc2):
        # Welford'un paralel merge formülü
        c1, m1, M2_1, mn1, mx1 = acc1
        c2, m2, M2_2, mn2, mx2 = acc2
        count = c1 + c2
        if count == 0:
            return (0, 0.0, 0.0, float('inf'), float('-inf'))
        delta = m2 - m1
        mean  = (m1 * c1 + m2 * c2) / count
        M2    = M2_1 + M2_2 + delta**2 * c1 * c2 / count
        return (count, mean, M2, min(mn1, mn2), max(mx1, mx2))

class AnomalyWindowProcessor(ProcessWindowFunction):
    """
    Pencere istatistiklerine göre z-score tabanlı anomali tespiti.
    z = (x - μ) / σ ; |z| > 3 ise anomali (3-sigma kuralı)
    """
    ZSCORE_THRESHOLD = 3.0

    def process(self, key, context, elements, out):
        for stats_json in elements:
            stats = json.loads(stats_json)
            mean    = stats['mean']
            std_dev = stats['std_dev']
            if std_dev < 0.001:  # Tüm fiyatlar eşit, anomali yok
                continue
            # z-score ile eşik kontrolü
            upper_bound = mean + self.ZSCORE_THRESHOLD * std_dev
            lower_bound = mean - self.ZSCORE_THRESHOLD * std_dev
            result = {
                'category':     key,
                'window_start': context.window().start,
                'window_end':   context.window().end,
                'mean':         mean,
                'std_dev':      std_dev,
                'upper_3sigma': round(upper_bound, 2),
                'lower_3sigma': round(lower_bound, 2),
                **stats
            }
            out.collect(json.dumps(result))

def anomaly_pipeline():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_parallelism(2)
    env.enable_checkpointing(15000)

    # 10 dakika uzunlukta, 2 dakikada bir kayan pencere
    stream = env.from_collection([], type_info=Types.STRING())

    result = stream \
        .key_by(lambda x: json.loads(x).get('category', 'unknown')) \
        .window(SlidingEventTimeWindows.of(
            Time.minutes(10),  # Pencere uzunluğu: L=10dk
            Time.minutes(2)    # Adım boyutu:     S=2dk
        )) \
        .aggregate(
            PriceStatsAggregator(),
            window_function=AnomalyWindowProcessor(),
            accumulator_type=Types.STRING(),
            output_type=Types.STRING()
        )

    result.print()
    env.execute('Price_Anomaly_Detection_Job')


### Çıktı Modları (Output Modes)

`bolum13/13_03_04_cikti-modlari.py`

_Kitap: Kod 13.4_


In [ ]:
# ─── Ön hazırlık ─────────────────────────────────────────────────────
# Bu kesim, kitapta bir önceki kesimde kurulan veriyi/modeli kullanır.
# Dosyanın tek başına çalışabilmesi için o hazırlık burada yinelenmiştir.
# Kaynak: PySpark tip tanımları
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType)
spark = (SparkSession.builder
         .appName("VM-ML Bolum13")
         .master("local[*]")
         .getOrCreate())
# ─── Ön hazırlık sonu ────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

spark = (SparkSession.builder
    .appName("KafkaStructuredStreaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# ─────────────────────────────────────────────────────────────────────
# 1. Kafka'dan Akan Veri Okuma
# ─────────────────────────────────────────────────────────────────────

kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "finansal-islemler")   # Çoklu: "topic1,topic2"
    .option("startingOffsets", "latest")         # "earliest" veya JSON offset
    .option("failOnDataLoss", "false")
    .option("maxOffsetsPerTrigger", 10000)        # Mikro-batch başına max mesaj
    .load())

# Kafka'dan gelen raw DataFrame şeması: key, value, topic, partition, offset, timestamp
kafka_df.printSchema()

# ─────────────────────────────────────────────────────────────────────
# 2. Mesaj Deserializasyonu
# ─────────────────────────────────────────────────────────────────────

txn_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("user_id",        StringType()),
    StructField("amount",         DoubleType()),
    StructField("merchant",       StringType()),
    StructField("country",        StringType()),
    StructField("timestamp",      StringType()),
    StructField("card_type",      StringType()),
])

# JSON mesajı parse et
parsed_df = (kafka_df
    .select(F.from_json(F.col("value").cast("string"), txn_schema).alias("data"),
            F.col("timestamp").alias("kafka_ts"))
    .select("data.*", "kafka_ts")
    .withColumn("event_time",
                F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"))
)

# ─────────────────────────────────────────────────────────────────────
# 3. Pencere Tabanlı Gerçek Zamanlı Aggregasyon
# ─────────────────────────────────────────────────────────────────────

# Watermark: 10 dakikaya kadar geç gelen verileri kabul et
windowed_agg = (parsed_df
    .withWatermark("event_time", "10 minutes")
    .groupBy(
        F.window(F.col("event_time"), "5 minutes"),   # 5 dakikalık pencere
        F.col("merchant")
    ).agg(
        F.count("transaction_id").alias("islem_sayisi"),
        F.sum("amount").alias("toplam_tutar"),
        F.avg("amount").alias("ort_tutar"),
        F.max("amount").alias("max_tutar"),
    ).orderBy("toplam_tutar", ascending=False)
)

# ─────────────────────────────────────────────────────────────────────
# 4. Gerçek Zamanlı Anomali Tespiti (UDF ile)
# ─────────────────────────────────────────────────────────────────────

# Kullanıcı başına 5 dakikadaki toplam harcama
user_window_agg = (parsed_df
    .withWatermark("event_time", "5 minutes")
    .groupBy(
        F.window(F.col("event_time"), "5 minutes", "1 minute"),  # Kayan pencere
        F.col("user_id")
    ).agg(
        F.sum("amount").alias("pencere_toplam"),
        F.count("*").alias("islem_adedi"),
    ).filter(F.col("pencere_toplam") > 3000)  # 5 dk'da 3000 TL üzeri uyarı
)

# ─────────────────────────────────────────────────────────────────────
# 5. ML Modeli ile Gerçek Zamanlı Sınıflandırma
# ─────────────────────────────────────────────────────────────────────

from pyspark.ml import PipelineModel

# Önceden eğitilmiş fraud detection modelini yükle
# fraud_model = PipelineModel.load("hdfs://cluster/models/fraud_detector_v3")

# Her mikro-batch'e model uygula (foreachBatch)
def process_batch(batch_df, batch_id):
    """Her mikro-batch üzerinde ML tahmini ve kayıt."""
    if batch_df.isEmpty():
        return
    print(f"Batch {batch_id}: {batch_df.count()} işlem işleniyor")
    # predictions = fraud_model.transform(batch_df)
    # predictions.filter(F.col("prediction") == 1) \
    #     .write.format("kafka") \
    #     .option("kafka.bootstrap.servers", "localhost:9092") \
    #     .option("topic", "fraud-alerts") \
    #     .save()
    batch_df.show(5)

# ─────────────────────────────────────────────────────────────────────
# 6. Sorguyu Başlatma ve Çıktı Yazma
# ─────────────────────────────────────────────────────────────────────

# Konsola yaz (geliştirme/test)
console_query = (windowed_agg.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="10 seconds")  # Her 10 saniyede bir mikro-batch
    .start())

# Parquet'a yaz (üretim)
# parquet_query = (windowed_agg.writeStream
#     .outputMode("append")
#     .format("parquet")
#     .option("path", "hdfs://cluster/streaming/merchant_stats")
#     .option("checkpointLocation", "hdfs://cluster/checkpoints/merchant_stats")
#     .trigger(processingTime="1 minute")
#     .start())

# foreachBatch ile ML model uygulama
ml_query = (parsed_df.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", "/tmp/checkpoints/fraud")
    .trigger(processingTime="5 seconds")
    .start())

# Tüm sorguları bekle
spark.streams.awaitAnyTermination(timeout=60)
spark.stop()


## 13.4. Akan Veri Üzerinde Makine Öğrenmesi Uygulamaları (Stream ML)


### 13.4.1. Akan Veride Sınıflandırma ve Kümeleme: Teorik Temeller ve Algoritmalar

`bolum13/13_04_01_akan-veride-siniflandirma-ve-kumeleme-teorik-tem.py`

_Kitap: Kod 13.5, Kod 13.6_


In [ ]:
import json, math
from pyflink.datastream.functions import KeyedProcessFunction
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.common.typeinfo import Types

class MicroClusterCF:
    """
    CluStream CF Vektörü: (N, LS, SS, LT, ST)
    Bellek: O(1) — nokta sayısından bağımsız.
    """
    __slots__ = ['n', 'ls', 'ss', 'lt', 'st', 'dim']

    def __init__(self, dim: int):
        self.n   = 0
        self.ls  = [0.0] * dim   # Doğrusal toplam (her boyut)
        self.ss  = [0.0] * dim   # Kareli toplam (her boyut)
        self.lt  = 0.0           # Zaman doğrusal toplamı
        self.st  = 0.0           # Zaman kareli toplamı
        self.dim = dim

    def add_point(self, x: list, t: float):
        """O(d) karmaşıklığıyla yeni nokta ekle."""
        self.n  += 1
        self.lt += t
        self.st += t * t
        for i, xi in enumerate(x):
            self.ls[i] += xi
            self.ss[i] += xi * xi

    @property
    def centroid(self) -> list:
        """Centroid = LS / N"""
        if self.n == 0:
            return [0.0] * self.dim
        return [ls_i / self.n for ls_i in self.ls]

    @property
    def radius(self) -> float:
        """
        Radius = sqrt(SS/N - (LS/N)^2)
        Öklid uzayında mikro-kümenin ortalama yarıçapı.
        """
        if self.n < 2:
            return 0.0
        centroid = self.centroid
        r_sq = sum(
            (self.ss[i] / self.n) - (centroid[i] ** 2)
            for i in range(self.dim)
        )
        return math.sqrt(max(r_sq, 0.0))

    def distance_to(self, x: list) -> float:
        """Noktanın centroid'e Öklid uzaklığı."""
        c = self.centroid
        return math.sqrt(sum((xi - ci)**2 for xi, ci in zip(x, c)))

    def to_dict(self) -> dict:
        return {
            'n':        self.n,
            'centroid': self.centroid,
            'radius':   round(self.radius, 4),
            'mean_time': self.lt / self.n if self.n > 0 else 0
        }

    @staticmethod
    def merge(cf1: 'MicroClusterCF', cf2: 'MicroClusterCF') -> 'MicroClusterCF':
        """İki CF vektörünü birleştir: Additive property."""
        merged = MicroClusterCF(cf1.dim)
        merged.n  = cf1.n  + cf2.n
        merged.lt = cf1.lt + cf2.lt
        merged.st = cf1.st + cf2.st
        merged.ls = [a + b for a, b in zip(cf1.ls, cf2.ls)]
        merged.ss = [a + b for a, b in zip(cf1.ss, cf2.ss)]
        return merged

class OnlineClusteringProcessor(KeyedProcessFunction):
    """
    Flink KeyedProcessFunction olarak CluStream Faz-1 (Online).
    Her anahtar (ör. ürün kategorisi) için bağımsız mikro-kümeler tutar.
    """
    MAX_MICRO_CLUSTERS = 10
    RADIUS_FACTOR      = 1.5  # Mevcut en yakın küme yarıçapının kaç katı

    def open(self, runtime_context):
        # Mikro-küme listesi state olarak saklanır
        desc = ValueStateDescriptor('micro_clusters', Types.STRING())
        self.cluster_state = runtime_context.get_state(desc)

    def process_element(self, value, ctx):
        event = json.loads(value)

        # Özellik vektörü çıkar
        features = [
            event.get('price', 0),
            event.get('quantity', 0),
            event.get('total_value', 0)
        ]
        t = event.get('timestamp', 0)

        # Mevcut mikro-kümeleri yükle
        raw = self.cluster_state.value()
        clusters_data = json.loads(raw) if raw else []
        clusters = []
        for cd in clusters_data:
            mc = MicroClusterCF(dim=3)
            mc.n, mc.ls, mc.ss, mc.lt, mc.st = (
                cd['n'], cd['ls'], cd['ss'], cd['lt'], cd['st']
            )
            clusters.append(mc)

        # En yakın mikro-kümeyi bul
        best_mc   = None
        best_dist = float('inf')
        for mc in clusters:
            d = mc.distance_to(features)
            if d < best_dist:
                best_dist = d
                best_mc   = mc

        # Atama kararı
        max_radius = (best_mc.radius * self.RADIUS_FACTOR
                      if best_mc and best_mc.radius > 0 else 1.0)
        if best_mc and best_dist <= max_radius:
            best_mc.add_point(features, t)  # Mevcut kümeye ekle
        elif len(clusters) < self.MAX_MICRO_CLUSTERS:
            new_mc = MicroClusterCF(dim=3)  # Yeni mikro-küme oluştur
            new_mc.add_point(features, t)
            clusters.append(new_mc)
        else:
            # En küçük kümeyi en yakın komşusuyla birleştir, yer aç
            smallest = min(clusters, key=lambda c: c.n)
            clusters.remove(smallest)
            if best_mc:
                merged = MicroClusterCF.merge(smallest, best_mc)
                clusters.remove(best_mc)
                clusters.append(merged)
            new_mc = MicroClusterCF(dim=3)
            new_mc.add_point(features, t)
            clusters.append(new_mc)

        # State'i güncelle
        clusters_data = [{'n': c.n, 'ls': c.ls, 'ss': c.ss, 'lt': c.lt, 'st': c.st}
                         for c in clusters]
        self.cluster_state.update(json.dumps(clusters_data))

        # Sonuç çıktısı
        result = {
            'category':      ctx.get_current_key(),
            'micro_clusters': [c.to_dict() for c in clusters],
            'timestamp':     t
        }
        yield json.dumps(result)


### 13.4.2. Gerçek Zamanlı Anomali Tespiti: Teorik Temeller ve Uygulama Mimarileri

`bolum13/13_04_02_gercek-zamanli-anomali-tespiti-teorik-temeller-v.py`


In [ ]:
import json, math, time
from collections import deque, defaultdict
from pyflink.datastream.functions import KeyedProcessFunction
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.common.typeinfo import Types

# ---- Online Welford İstatistik Takipçisi ----
class WelfordOnlineStats:
    """
    Welford'un tek geçişli online ortalama ve varyans algoritması.
    Bellek: O(1) — tüm geçmiş veri noktaları depolanmaz.
    Güncelleme karmaşıklığı: O(1)
    """
    def __init__(self):
        self.n     = 0
        self.mean  = 0.0
        self.M2    = 0.0  # Varyansın birikimli karesi

    def update(self, x: float):
        self.n    += 1
        delta      = x - self.mean
        self.mean += delta / self.n
        delta2     = x - self.mean
        self.M2   += delta * delta2

    @property
    def variance(self) -> float:
        return self.M2 / (self.n - 1) if self.n > 1 else 0.0

    @property
    def std_dev(self) -> float:
        return math.sqrt(self.variance)

    def z_score(self, x: float) -> float:
        """z = (x - μ) / σ; |z| > 3 ise anomali."""
        if self.std_dev < 1e-9:
            return 0.0
        return (x - self.mean) / self.std_dev

# ---- CUSUM Değişim Noktası Dedektörü ----
class CUSUMDetector:
    """
    C_t = max(0, C_{t-1} + (x_t - mu0 - k))
    Alarm: C_t > H
    """
    def __init__(self, mu0: float = 0.0, k: float = 0.5, H: float = 5.0):
        self.mu0 = mu0   # Referans ortalama
        self.k   = k     # Slack (izin verilen sapma)
        self.H   = H     # Alarm eşiği
        self.C   = 0.0   # Kümülatif toplam

    def update(self, x: float) -> bool:
        self.C = max(0.0, self.C + (x - self.mu0 - self.k))
        return self.C > self.H  # True ise alarm

    def reset(self):
        self.C = 0.0

# ---- Kural Tabanlı Filtre ----
class RuleBasedFilter:
    """Hız, coğrafya ve tutar tabanlı deterministik kurallar."""

    def check(self, event: dict, history: deque) -> list:
        alerts = []
        price  = event.get('price', 0)

        # Kural 1: Aşırı yüksek tutar
        if price > 10000:
            alerts.append(('HIGH_AMOUNT', 0.8))

        # Kural 2: Hız kontrolü — 60sn içinde 5+ işlem
        now = event.get('timestamp', time.time())
        recent = [e for e in history if now - e.get('timestamp', 0) < 60]
        if len(recent) >= 5:
            alerts.append(('HIGH_VELOCITY', 0.9))

        # Kural 3: Daha önce hiç görmediğimiz yüksek riskli ülke
        HIGH_RISK = {'NG', 'RU', 'CN', 'PK'}
        if event.get('country', '') in HIGH_RISK and price > 500:
            alerts.append(('HIGH_RISK_COUNTRY', 0.7))

        return alerts

# ---- Ana Fraud Dedektörü (Flink KeyedProcessFunction) ----
class StreamingFraudDetector(KeyedProcessFunction):
    """
    Çok katmanlı fraud tespiti:
    1. Kural katmanı (deterministik)
    2. İstatistiksel katman (z-score, Welford)
    3. CUSUM değişim noktası tespiti
    Son karar: ağırlıklı ensemble skoru
    """
    FRAUD_SCORE_THRESHOLD = 0.6   # Bu değer üstü = fraud şüphesi
    HISTORY_SIZE          = 20    # Son 20 işlem hafızada

    def open(self, runtime_context):
        # Welford istatistik state'i (JSON serialized)
        self.stats_state = runtime_context.get_state(
            ValueStateDescriptor('welford_stats', Types.STRING())
        )
        # İşlem geçmişi (son N işlem)
        self.history_state = runtime_context.get_state(
            ValueStateDescriptor('tx_history', Types.STRING())
        )
        # CUSUM state
        self.cusum_state = runtime_context.get_state(
            ValueStateDescriptor('cusum_C', Types.DOUBLE())
        )
        # Kalıcı sınıf örnekleri (state değil, in-memory helper)
        self.rule_filter = RuleBasedFilter()

    def _load_stats(self) -> WelfordOnlineStats:
        raw = self.stats_state.value()
        ws  = WelfordOnlineStats()
        if raw:
            d = json.loads(raw)
            ws.n, ws.mean, ws.M2 = d['n'], d['mean'], d['M2']
        return ws

    def _save_stats(self, ws: WelfordOnlineStats):
        self.stats_state.update(json.dumps({'n': ws.n, 'mean': ws.mean, 'M2': ws.M2}))

    def process_element(self, value, ctx):
        event    = json.loads(value)
        price    = event.get('price', 0)
        user_id  = ctx.get_current_key()

        # ---- Geçmişi yükle ----
        raw_hist  = self.history_state.value()
        history   = deque(json.loads(raw_hist) if raw_hist else [], maxlen=self.HISTORY_SIZE)

        # ---- Katman 1: Kural bazlı ----
        rule_alerts = self.rule_filter.check(event, history)
        rule_score  = max((score for _, score in rule_alerts), default=0.0)

        # ---- Katman 2: İstatistiksel (Welford z-score) ----
        ws          = self._load_stats()
        z_score     = ws.z_score(price)
        stat_score  = min(abs(z_score) / 5.0, 1.0)  # Normalize [0,1]
        ws.update(price)  # Modeli güncelle
        self._save_stats(ws)

        # ---- Katman 3: CUSUM ----
        cusum_C      = self.cusum_state.value() or 0.0
        cusum        = CUSUMDetector(mu0=ws.mean, k=ws.std_dev * 0.5, H=5.0)
        cusum.C      = cusum_C
        cusum_alarm  = cusum.update(price)
        cusum_score  = 0.7 if cusum_alarm else 0.0
        self.cusum_state.update(cusum.C if not cusum_alarm else 0.0)

        # ---- Ensemble: Ağırlıklı Ortalama ----
        final_score = (0.4 * rule_score + 0.35 * stat_score + 0.25 * cusum_score)

        result = {
            **event,
            'fraud_score':    round(final_score, 4),
            'rule_score':     round(rule_score, 4),
            'stat_score':     round(stat_score, 4),
            'z_score':        round(z_score, 4),
            'cusum_score':    round(cusum_score, 4),
            'rule_alerts':    [name for name, _ in rule_alerts],
            'is_fraud_alert': final_score >= self.FRAUD_SCORE_THRESHOLD
        }

        # Geçmişi güncelle
        history.append(event)
        self.history_state.update(json.dumps(list(history)))

        yield json.dumps(result)

# ================================================================
# River — Python Online Machine Learning Kütüphanesi
# pip install river
# Referans: Online ML için scikit-multiflow'dan daha modern alternatif
# ================================================================
from river import (
    stream as rv_stream,
    tree,
    ensemble,
    anomaly,
    drift,
    metrics,
    preprocessing
)
import json

# ---- Online Hoeffding Tree Sınıflandırıcı ----
class OnlineStreamClassifier:
    """
    River'ın Hoeffding Tree sınıflandırıcısını kullanan online model.
    Her örnek görüldükten sonra model güncellenir (learn_one).
    ADWIN ile konsept kayması izlenir.
    """
    def __init__(self):
        # Hoeffding Adaptive Tree: konsept kaymasına yerleşik adaptasyon
        self.model = ensemble.AdaptiveRandomForestClassifier(
            n_models=10,
            seed=42
        )
        # Konsept Kayması Detektörü
        self.drift_detector = drift.ADWIN(delta=0.002)
        # Performans metrikleri
        self.accuracy  = metrics.Accuracy()
        self.kappa     = metrics.CohenKappa()
        self.n_samples = 0
        self.n_drifts  = 0

    def predict_and_update(self, x: dict, y: int) -> dict:
        """
        1. Tahmin yap (predict_one)
        2. Gerçek etiketi öğren (learn_one)
        3. ADWIN ile konsept kayması kontrol et
        """
        # Tahmin
        pred = self.model.predict_one(x)
        prob = self.model.predict_proba_one(x)

        # Öğren
        self.model.learn_one(x, y)
        self.n_samples += 1

        # Metrik güncelle
        if pred is not None:
            self.accuracy.update(y, pred)
            self.kappa.update(y, pred)

        # ADWIN: konsept kayması tespiti
        correct = int(pred == y) if pred is not None else 0
        self.drift_detector.update(correct)
        drift_detected = self.drift_detector.drift_detected
        if drift_detected:
            self.n_drifts += 1

        return {
            'prediction':    pred,
            'probability':   prob,
            'accuracy':      self.accuracy.get(),
            'kappa':         self.kappa.get(),
            'drift_detected': drift_detected,
            'n_drifts':      self.n_drifts,
            'n_samples':     self.n_samples
        }

# ---- Kullanım: Simüle Edilmiş Fraud Akışı ----
def simulate_streaming_fraud_detection():
    from river.datasets import CreditCard

    classifier = OnlineStreamClassifier()

    print(f"{'Örnek':>8} | {'Tahmin':>7} | {'Gerçek':>7} | {'Doğruluk':>9} | {'Kayma':>6}")
    print('-' * 55)

    # Streaming veri simülasyonu (her örnek tek seferde işlenir)
    for i, (x, y) in enumerate(CreditCard().take(10000)):

        # Özellik ön-işleme (online normalleştirme)
        result = classifier.predict_and_update(x, y)

        if i % 500 == 0:
            print(
                f"{i:>8} | {result['prediction']:>7} | {y:>7} | "
                f"{result['accuracy']:>9.4f} | {result['n_drifts']:>6}"
            )

    print(f"\nFinal Doğruluk: {classifier.accuracy.get():.4f}")
    print(f"Toplam Konsept Kayması Sayısı: {classifier.n_drifts}")
    print(f"Cohen Kappa: {classifier.kappa.get():.4f}")

if __name__ == '__main__':
    simulate_streaming_fraud_detection()


## 13.5. Uçtan Uca Vaka Çalışması (Case Study):


### 13.5.2. Veri Üretimi: Tweepy ile Canlı Twitter Akışı ve Kafka'ya İletim

`bolum13/13_05_02_veri-uretimi-tweepy-ile-canli-twitter-akisi-ve-k.py`

_Kitap: Kod 13.7_


In [ ]:
import tweepy
import json
import re
import time
import logging
from kafka import KafkaProducer
from kafka.errors import KafkaError

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ---- Tweet Temizleme Yardımcı Fonksiyon ----
def clean_tweet(text: str) -> str:
    """
    NLP modellerinin daha iyi sonuç vermesi için tweet metnini temizler.
    Sıralı dönüşümler:
    1. URL'ler kaldırılır
    2. @mention'lar kaldırılır
    3. RT (retweet) öneki kaldırılır
    4. Çift boşluklar tekleştirilir
    5. Baştaki/sondaki boşluklar temizlenir
    """
    text = re.sub(r'http\S+|www\.\S+', '', text)       # URL'ler\ntext = re.sub(r'@\w+', '', text)                    # Mention'lar
    text = re.sub(r'^RT\s?:', '', text)                 # Retweet öneki
    text = re.sub(r'#', '', text)                        # Hashtag sembolü (#AI → AI)
    text = re.sub(r'\s+', ' ', text)                    # Çift boşluklar
    return text.strip()

# ---- Kafka Producer Fabrikası ----
def create_producer(bootstrap_servers: list) -> KafkaProducer:
    return KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8'),
        key_serializer=lambda k: k.encode('utf-8') if k else None,
        acks='all',
        retries=5,
        linger_ms=10,
        compression_type='lz4'
    )

# ---- Tweepy StreamingClient Alt Sınıfı ----
class TweetStreamToKafka(tweepy.StreamingClient):
    """
    Tweepy StreamingClient'ı genişleterek gelen her tweet'i
    Kafka'ya iletir.
    """
    TOPIC_NAME     = 'tweet_stream'
    TARGET_LANGS   = {'en', 'tr'}  # Hangi dilleri işleyeceğiz

    def __init__(self, bearer_token: str, kafka_servers: list, **kwargs):
        super().__init__(bearer_token, **kwargs)
        self.producer    = create_producer(kafka_servers)
        self.tweet_count = 0
        self.error_count = 0

    def on_tweet(self, tweet):
        """Her yeni tweet geldiğinde otomatik olarak çağrılır."""
        try:
            # Dil filtresi: sadece hedef dilleri işle
            lang = getattr(tweet, 'lang', 'und')
            if lang not in self.TARGET_LANGS:
                return

            # Hashtag listesi çıkar
            hashtags = []
            if hasattr(tweet, 'entities') and tweet.entities:
                ht_list = tweet.entities.get('hashtags', [])
                hashtags = [ht['tag'].lower() for ht in ht_list]

            # Metrik bilgileri
            metrics = {}
            if hasattr(tweet, 'public_metrics') and tweet.public_metrics:
                metrics = tweet.public_metrics

            # Kafka'ya gönderilecek payload oluştur
            payload = {
                'tweet_id':      str(tweet.id),
                'text':          tweet.text,
                'clean_text':    clean_tweet(tweet.text),
                'lang':          lang,
                'author_id':     str(tweet.author_id) if tweet.author_id else None,
                'hashtags':      hashtags,
                'retweet_count': metrics.get('retweet_count', 0),
                'like_count':    metrics.get('like_count', 0),
                # Event Time: tweet'in gerçek oluşturulma zamanı
                'created_at':    tweet.created_at.isoformat() if tweet.created_at else None,
                # Ingestion Time: Kafka'ya girdiği an
                'ingestion_ts':  time.time()
            }

            # Partition key: dil kodu (aynı dil → aynı partition → sıra garantisi)
            self.producer.send(
                self.TOPIC_NAME,
                key=lang,
                value=payload
            )
            self.tweet_count += 1

            if self.tweet_count % 100 == 0:
                logger.info(f"Kafka'ya gönderilen tweet: {self.tweet_count}")

        except Exception as e:
            self.error_count += 1
            logger.error(f'Tweet işleme hatası: {e}')

    def on_error(self, status_code):
        logger.error(f'Twitter API Hatası: {status_code}')
        if status_code == 429:  # Rate limit
            logger.warning('Rate limit aşıldı. 60 saniye bekleniyor...')
            time.sleep(60)
        return True  # True: bağlantıyı koru

    def on_disconnect(self):
        logger.warning('Twitter bağlantısı kesildi. Yeniden bağlanılıyor...')
        self.producer.flush()

# ---- Filtreleme Kuralları Yönetimi ----
def setup_stream_rules(client: TweetStreamToKafka, keywords: list):
    """
    Mevcut kuralları temizle ve yeni kurallar ekle.
    Twitter API v2 kural dili örnekleri:
    - 'python lang:en -is:retweet'  → İngilizce python tweet'leri, RT hariç
    - '#AI OR #MachineLearning'      → Bu hashtag'lerden biri olan tweet'ler
    - 'yapay zeka lang:tr'           → Türkçe yapay zeka tweet'leri
    """
    # Mevcut kuralları sil
    existing = client.get_rules().data
    if existing:
        rule_ids = [rule.id for rule in existing]
        client.delete_rules(rule_ids)
        logger.info(f'{len(rule_ids)} eski kural silindi.')

    # Yeni kurallar ekle
    new_rules = []
    for kw in keywords:
        rule_text = f'{kw} -is:retweet lang:en OR lang:tr'
        new_rules.append(tweepy.StreamRule(rule_text))
    client.add_rules(new_rules)
    logger.info(f'{len(new_rules)} yeni kural eklendi: {keywords}')

# ---- Ana Çalıştırıcı ----
def run_twitter_stream(
    bearer_token: str,
    kafka_servers: list,
    keywords: list
):
    stream = TweetStreamToKafka(
        bearer_token=bearer_token,
        kafka_servers=kafka_servers,
        wait_on_rate_limit=True  # Rate limit'e çarparsa otomatik bekle
    )
    setup_stream_rules(stream, keywords)

    logger.info(f'Twitter akışı başlatılıyor. Anahtar kelimeler: {keywords}')
    stream.filter(
        tweet_fields=['created_at', 'lang', 'author_id', 'public_metrics'],
        expansions=['entities.mentions.username'],
        media_fields=['url']
    )

# Gerçek kullanım:
# run_twitter_stream(
#     bearer_token='YOUR_BEARER_TOKEN',
#     kafka_servers=['localhost:9092'],
#     keywords=['yapay zeka', 'artificial intelligence', '#AI']
# )

# ================================================================
# Gerçekçi Tweet Simülatörü — Production-grade test ortamı
# ================================================================
import json, time, random, uuid, re
from datetime import datetime, timezone
from kafka import KafkaProducer
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('TweetSimulator')

# Gerçekçi tweet şablonları (çeşitli duygu tonları)
TWEET_TEMPLATES = {
    'positive': [
        '{topic} gerçekten harika! Bu teknolojiyi çok seviyorum. #innovation',
        'Bugün {topic} hakkında muhteşem bir şey öğrendim. Geleceğe umutla bakıyorum!',
        '{topic} ile yapılan bu proje inanılmaz sonuçlar veriyor. Tebrikler ekibe!',
        'Finally understood {topic} and it is absolutely amazing! #tech #AI',
        '{topic} is revolutionizing everything. Cant wait to see whats next!'
    ],
    'negative': [
        '{topic} hâlâ pek çok sorunu çözemiyor. Hayal kırıklığı büyük.',
        'Bu {topic} ürünü tam bir hayal kırıklığı. Param çöpe gitti.',
        '{topic} ile ilgili bu gelişme endişe verici. Kimse önlem almıyor.',
        'Disappointed with {topic}. Expected much more. #fail',
        '{topic} is overhyped and underdelivering. Not impressed at all.'
    ],
    'neutral': [
        '{topic} hakkında yeni bir rapor yayınlandı. İnceliyorum.',
        'Bugün {topic} konferansına katıldım. Notlar hazırlıyorum.',
        '{topic} trends are changing rapidly. Interesting to observe.',
        'New study about {topic} published today. Worth reading.',
        '{topic} market update: mixed signals from analysts.'
    ]
}

TOPICS = ['yapay zeka', 'artificial intelligence', 'machine learning',
          'deep learning', 'ChatGPT', 'data science', 'Python', 'Flink']

HASHTAG_POOL = {
    'tech': ['AI', 'MachineLearning', 'DataScience', 'DeepLearning', 'Python'],
    'business': ['Innovation', 'Tech', 'Startup', 'Digital', 'Future'],
    'community': ['OpenSource', 'Developer', 'Programming', 'TechTwitter']
}

LANGUAGES = ['en', 'en', 'en', 'tr', 'tr']  # İngilizce ağırlıklı

def generate_mock_tweet(topic: str = None) -> dict:
    """
    Gerçekçi duygu dağılımı simüle eden tweet üretir:
    Pozitif: %45, Negatif: %30, Nötr: %25
    (Gerçek Twitter duygu dağılımına yakın değerler)
    """
    if topic is None:
        topic = random.choice(TOPICS)

    # Gerçekçi duygu dağılımı
    rand = random.random()
    if rand < 0.45:
        sentiment = 'positive'
    elif rand < 0.75:
        sentiment = 'negative'
    else:
        sentiment = 'neutral'

    lang = random.choice(LANGUAGES)

    # Tweet metni oluştur
    template = random.choice(TWEET_TEMPLATES[sentiment])
    text = template.format(topic=topic)

    # Hashtag'ler ekle
    category = random.choice(list(HASHTAG_POOL.keys()))
    hashtags = random.sample(HASHTAG_POOL[category], k=random.randint(1, 3))

    # Gerçekçi engagement metrikleri (uzun kuyruklu dağılım)
    follower_weight = random.paretovariate(1.5)
    retweet_count   = int(random.expovariate(0.1) * follower_weight)
    like_count      = int(retweet_count * random.uniform(2, 10))

    return {
        'tweet_id':      str(uuid.uuid4()),
        'text':          text + ' ' + ' '.join(f'#{h}' for h in hashtags),
        'clean_text':    re.sub(r'#\w+', '', text).strip(),
        'lang':          lang,
        'author_id':     f'user_{random.randint(10000, 9999999)}',
        'hashtags':      hashtags,
        'retweet_count': retweet_count,
        'like_count':    like_count,
        'created_at':    datetime.now(timezone.utc).isoformat(),
        'ingestion_ts':  time.time(),
        # Ground truth label (simülatör bilir, gerçekte yoktur)
        '_sim_sentiment': sentiment
    }

def run_mock_producer(
    kafka_servers: list,
    topic_name: str = 'tweet_stream',
    tweets_per_second: float = 50.0,
    burst_mode: bool = False
):
    """
    Ayarlanabilir hızda tweet akışı simüle eder.
    burst_mode=True ile zirve trafik simülasyonu yapılabilir.
    """
    producer = KafkaProducer(
        bootstrap_servers=kafka_servers,
        value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8'),
        key_serializer=lambda k: k.encode('utf-8'),
        compression_type='lz4',
        linger_ms=5
    )

    sleep_interval = 1.0 / tweets_per_second
    sent   = 0
    topic  = random.choice(TOPICS)

    logger.info(f'Mock Producer: {tweets_per_second} tweet/sn hızında başlatıldı.')

    try:
        while True:
            # Burst modu: her 60sn'de 10sn boyunca 10x trafik
            if burst_mode and sent % 3000 == 2999:
                logger.info('BURST MODE başlıyor — 10 saniye yüksek trafik!')
                for _ in range(int(tweets_per_second * 10 * 10)):  # 10x * 10sn
                    tweet = generate_mock_tweet()
                    producer.send(topic_name, key=tweet['lang'], value=tweet)
                logger.info('BURST MODE bitti.')
                producer.flush()

            tweet = generate_mock_tweet(topic=topic)
            producer.send(topic_name, key=tweet['lang'], value=tweet)
            sent += 1

            # Her 1000 tweet'te topic değiştir (trending topic simülasyonu)
            if sent % 1000 == 0:
                topic = random.choice(TOPICS)
                logger.info(f'Gönderilen: {sent} | Güncel topic: {topic}')
                producer.flush()

            time.sleep(sleep_interval)

    except KeyboardInterrupt:
        logger.info(f'Simülatör durduruldu. Toplam: {sent} tweet.')
    finally:
        producer.flush()
        producer.close()

if __name__ == '__main__':
    run_mock_producer(
        kafka_servers=['localhost:9092'],
        tweets_per_second=50,  # Saniyede 50 tweet
        burst_mode=True        # Zirve trafik testi
    )


### 13.5.3. PyFlink ile Canlı Analiz: NLP Entegrasyonu ve Duygu Analizi Pipeline'ı

`bolum13/13_05_03_pyflink-ile-canli-analiz-nlp-entegrasyonu-ve-duy.py`

_Kitap: Kod 13.8, Kod 13.9_


In [ ]:
import json, time, re, logging
from pyflink.datastream import StreamExecutionEnvironment, TimeCharacteristic
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer, FlinkKafkaProducer
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
from pyflink.datastream.functions import (
    MapFunction, FilterFunction, FlatMapFunction,
    AggregateFunction, ProcessWindowFunction
)
from pyflink.datastream.window import TumblingEventTimeWindows, SlidingEventTimeWindows
from pyflink.common.time import Time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('SentimentPipeline')

# ================================================================
# KATMAN 1: Ön İşleme
# ================================================================
class TweetPreprocessor(MapFunction):
    """
    Tweet metnini NLP modellerine hazırlar.
    Akış içinde her olay için O(len(text)) karmaşıklıkta çalışır.
    """
    URL_PATTERN     = re.compile(r'http\S+|www\.\S+')
    MENTION_PATTERN = re.compile(r'@\w+')
    EMOJI_PATTERN   = re.compile(
        '[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
        '\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+',
        flags=re.UNICODE
    )

    def map(self, value: str) -> str:
        try:
            tweet = json.loads(value)
            text = tweet.get('text', '')

            # Temizleme zinciri
            text = self.URL_PATTERN.sub('', text)
            text = self.MENTION_PATTERN.sub('', text)
            # Emoji'leri koru (VADER emoji'lerden anlam çıkarır)
            text = re.sub(r'\s+', ' ', text).strip()

            tweet['processed_text'] = text
            tweet['char_count']     = len(text)
            tweet['word_count']     = len(text.split())
            return json.dumps(tweet, ensure_ascii=False)
        except Exception as e:
            logger.warning(f'Önişleme hatası: {e}')
            return value  # Hatalı kayıtlar değişmeden geçsin

class ValidTweetFilter(FilterFunction):
    """Kısa, boş veya dil dışı tweet'leri elee."""
    MIN_WORDS = 3
    ALLOWED_LANGS = {'en', 'tr'}

    def filter(self, value: str) -> bool:
        try:
            tweet = json.loads(value)
            text  = tweet.get('processed_text', '')
            lang  = tweet.get('lang', 'und')
            return (
                lang in self.ALLOWED_LANGS and
                len(text.split()) >= self.MIN_WORDS
            )
        except Exception:
            return False

# ================================================================
# KATMAN 2: Duygu Analizi (VADER birincil + DistilBERT ikincil)
# ================================================================
class HybridSentimentAnalyzer(MapFunction):
    """
    İki aşamalı hibrit NLP sistemi.
    VADER hızlı ama yüzeysel; DistilBERT yavaş ama derin.
    Güven eşiğine göre hangi modelin kullanılacağı karar verilir.
    """
    VADER_CONFIDENCE_THRESHOLD = 0.5   # Bu değer üstünde VADER'a güven
    BERT_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'

    def open(self, runtime_context):
        # VADER'ı başlat (senkron, hızlı)
        import nltk
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        self.vader = SentimentIntensityAnalyzer()

        # DistilBERT'i başlat (asenkron yükleme, zaman alır)
        try:
            from transformers import pipeline as hf_pipeline
            self.bert_pipeline = hf_pipeline(
                'sentiment-analysis',
                model=self.BERT_MODEL_NAME,
                device=-1,     # -1: CPU, 0: GPU (varsa)
                truncation=True,
                max_length=128  # Tweet'ler genelde kısadır
            )
            self.bert_available = True
            logger.info('DistilBERT başarıyla yüklendi.')
        except Exception as e:
            logger.warning(f'DistilBERT yüklenemedi, yalnızca VADER kullanılacak: {e}')
            self.bert_available = False

    def _vader_analyze(self, text: str) -> dict:
        scores = self.vader.polarity_scores(text)
        compound = scores['compound']
        label = ('POSITIVE' if compound >= 0.05
                 else 'NEGATIVE' if compound <= -0.05
                 else 'NEUTRAL')
        confidence = abs(compound)
        return {
            'vader_compound': compound,
            'vader_pos':      scores['pos'],
            'vader_neg':      scores['neg'],
            'vader_neu':      scores['neu'],
            'vader_label':    label,
            'vader_confidence': confidence
        }

    def _bert_analyze(self, text: str) -> dict:
        try:
            result = self.bert_pipeline(text[:512])[0]  # Truncate
            label  = result['label']  # 'POSITIVE' or 'NEGATIVE'
            score  = result['score']  # [0, 1]
            return {
                'bert_label':  label,
                'bert_score':  score,
                'bert_compound': score if label == 'POSITIVE' else -score
            }
        except Exception as e:
            return {'bert_label': 'NEUTRAL', 'bert_score': 0.5, 'bert_compound': 0.0}

    def map(self, value: str) -> str:
        try:
            tweet = json.loads(value)
            text  = tweet.get('processed_text', '')

            # Aşama 1: VADER analizi
            vader_result = self._vader_analyze(text)
            tweet.update(vader_result)

            # Aşama 2: Düşük güven → DistilBERT'e yönlendir
            used_bert = False
            if (self.bert_available and
                    vader_result['vader_confidence'] < self.VADER_CONFIDENCE_THRESHOLD):
                bert_result = self._bert_analyze(text)
                tweet.update(bert_result)
                used_bert = True

                # Hibrit skor: 0.35 * VADER + 0.65 * BERT
                final_compound = (
                    0.35 * vader_result['vader_compound'] +
                    0.65 * bert_result['bert_compound']
                )
            else:
                final_compound = vader_result['vader_compound']

            # Final sınıflandırma
            tweet['final_compound'] = round(final_compound, 4)
            tweet['final_label'] = (
                'POSITIVE' if final_compound >= 0.05
                else 'NEGATIVE' if final_compound <= -0.05
                else 'NEUTRAL'
            )
            tweet['used_bert']   = used_bert
            tweet['analyzed_at'] = time.time()

            return json.dumps(tweet, ensure_ascii=False)
        except Exception as e:
            logger.error(f'Sentiment analiz hatası: {e}')
            return value

# ================================================================
# KATMAN 3: Hashtag Çıkarıcı (FlatMap — Bire-Çok)
# ================================================================
class HashtagExpander(FlatMapFunction):
    """
    Her tweet için, içerdiği hashtag başına bir kayıt üretir.
    Bu, hashtag bazında aggregation yapabilmek için gereklidir.
    Bire-Çok (FlatMap) dönüşümü: 1 tweet → N hashtag kaydı
    """
    def flat_map(self, value: str):
        try:
            tweet = json.loads(value)
            hashtags = tweet.get('hashtags', [])

            if not hashtags:
                # Hashtag yoksa '#general' olarak etiketle
                record = dict(tweet)
                record['hashtag_key'] = '_general'
                yield json.dumps(record)
                return

            for tag in hashtags:
                record = dict(tweet)
                record['hashtag_key'] = tag.lower()
                yield json.dumps(record)
        except Exception:
            yield value

# ================================================================
# KATMAN 4: Pencere Tabanlı Hashtag Duygu Agregasyonu
# ================================================================
class SentimentAccumulator:
    """Pencere içindeki duygu istatistiklerini tutar."""
    __slots__ = ['count', 'pos', 'neg', 'neu', 'sum_compound',
                 'sum_likes', 'sum_rts']
    def __init__(self):
        self.count       = 0
        self.pos         = 0
        self.neg         = 0
        self.neu         = 0
        self.sum_compound = 0.0
        self.sum_likes   = 0
        self.sum_rts     = 0

class HashtagSentimentAggregator(AggregateFunction):
    """
    Kayan/Atlamalı pencerede hashtag bazında duygu istatistikleri:
    - Ortalama duygu skoru (mean compound)
    - Duygu dağılımı (pos/neg/neu yüzdeleri)
    - Etkileşim ağırlıklı skor (engagement-weighted)
    """
    def create_accumulator(self):
        return SentimentAccumulator()

    def add(self, value: str, acc: SentimentAccumulator):
        try:
            tweet = json.loads(value)
            label = tweet.get('final_label', 'NEUTRAL')
            acc.count += 1
            acc.sum_compound += tweet.get('final_compound', 0.0)
            acc.sum_likes    += tweet.get('like_count', 0)
            acc.sum_rts      += tweet.get('retweet_count', 0)
            if label == 'POSITIVE': acc.pos += 1
            elif label == 'NEGATIVE': acc.neg += 1
            else: acc.neu += 1
        except Exception:
            pass
        return acc

    def get_result(self, acc: SentimentAccumulator) -> str:
        n = max(acc.count, 1)
        mean_compound = acc.sum_compound / n
        # Etkileşim ağırlıklı skor:
        # engagement = log(1 + likes + 3*retweets)  [RT daha değerli]
        import math
        engagement_weight = math.log1p(acc.sum_likes + 3 * acc.sum_rts)
        weighted_score    = mean_compound * (1 + 0.1 * engagement_weight)
        return json.dumps({
            'tweet_count':        acc.count,
            'mean_compound':      round(mean_compound, 4),
            'engagement_score':   round(min(weighted_score, 1.0), 4),
            'positive_pct':       round(acc.pos / n * 100, 1),
            'negative_pct':       round(acc.neg / n * 100, 1),
            'neutral_pct':        round(acc.neu / n * 100, 1),
            'total_likes':        acc.sum_likes,
            'total_retweets':     acc.sum_rts,
            'dominant_sentiment': ('POSITIVE' if acc.pos >= acc.neg and acc.pos >= acc.neu
                                   else 'NEGATIVE' if acc.neg >= acc.pos and acc.neg >= acc.neu
                                   else 'NEUTRAL')
        })

    def merge(self, acc1: SentimentAccumulator, acc2: SentimentAccumulator):
        acc1.count        += acc2.count
        acc1.pos          += acc2.pos
        acc1.neg          += acc2.neg
        acc1.neu          += acc2.neu
        acc1.sum_compound += acc2.sum_compound
        acc1.sum_likes    += acc2.sum_likes
        acc1.sum_rts      += acc2.sum_rts
        return acc1

class WindowedResultEnricher(ProcessWindowFunction):
    """Pencere metadata'sını sonuca ekler."""
    def process(self, key, context, elements, out):
        for agg_json in elements:
            agg = json.loads(agg_json)
            agg['hashtag']      = key
            agg['window_start'] = context.window().start // 1000  # ms → saniye
            agg['window_end']   = context.window().end   // 1000
            agg['window_size']  = '5min_tumbling'
            out.collect(json.dumps(agg))

# ================================================================
# MAIN: Pipeline Tanımı ve Çalıştırma
# ================================================================
def build_sentiment_pipeline(
    kafka_servers: str = 'localhost:9092',
    source_topic:  str = 'tweet_stream',
    result_topic:  str = 'sentiment_results',
    aggreg_topic:  str = 'hashtag_aggregations',
    parallelism:   int = 4,
    checkpoint_interval_ms: int = 30_000
):
    # --- Flink Ortamı ---
    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_stream_time_characteristic(TimeCharacteristic.EventTime)
    env.set_parallelism(parallelism)
    env.enable_checkpointing(checkpoint_interval_ms)

    # --- Kafka Source ---
    kafka_props = {
        'bootstrap.servers': kafka_servers,
        'group.id':          'flink_sentiment_group'
    }
    kafka_source = FlinkKafkaConsumer(
        topics=source_topic,
        deserialization_schema=SimpleStringSchema(),
        properties=kafka_props
    )
    kafka_source.set_start_from_latest()

    # --- Pipeline Zinciri ---
    raw_stream = env.add_source(kafka_source)

    # Ön işleme ve filtreleme
    clean_stream = (
        raw_stream
        .map(TweetPreprocessor(), output_type=Types.STRING())
        .filter(ValidTweetFilter())
    )

    # Duygu analizi (her tweet için bireysel skor)
    analyzed_stream = clean_stream.map(
        HybridSentimentAnalyzer(), output_type=Types.STRING()
    )

    # Bireysel sonuçları Kafka'ya yaz
    analyzed_stream.add_sink(FlinkKafkaProducer(
        topic=result_topic,
        serialization_schema=SimpleStringSchema(),
        producer_config={'bootstrap.servers': kafka_servers}
    ))

    # Hashtag bazında 5 dakikalık atlamalı pencere agregasyonu
    hashtag_agg = (
        analyzed_stream
        .flat_map(HashtagExpander(), output_type=Types.STRING())
        .key_by(lambda x: json.loads(x).get('hashtag_key', '_general'))
        .window(TumblingEventTimeWindows.of(Time.minutes(5)))
        .aggregate(
            HashtagSentimentAggregator(),
            window_function=WindowedResultEnricher(),
            accumulator_type=Types.STRING(),
            output_type=Types.STRING()
        )
    )

    hashtag_agg.add_sink(FlinkKafkaProducer(
        topic=aggreg_topic,
        serialization_schema=SimpleStringSchema(),
        producer_config={'bootstrap.servers': kafka_servers}
    ))

    # Konsola yazdır (debug için)
    analyzed_stream.print()

    logger.info('PyFlink Sentiment Pipeline baslatiliyor...')
    env.execute('Twitter_Realtime_Sentiment_Analysis')

if __name__ == '__main__':
    build_sentiment_pipeline()

# ================================================================
# Elasticsearch Sink — Flink sonuçlarını ES'e yaz
# pip install elasticsearch
# ================================================================
from elasticsearch import Elasticsearch, helpers
from kafka import KafkaConsumer
import json, time, logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('ESSink')

# Elasticsearch Index Mapping (Kibana için optimize edilmiş)
TWEET_MAPPING = {
    'mappings': {
        'properties': {
            'tweet_id':       {'type': 'keyword'},
            'text':           {'type': 'text', 'analyzer': 'english'},
            'processed_text': {'type': 'text', 'analyzer': 'english'},
            'lang':           {'type': 'keyword'},
            'hashtags':       {'type': 'keyword'},
            'final_compound': {'type': 'float'},
            'final_label':    {'type': 'keyword'},
            'vader_compound': {'type': 'float'},
            'used_bert':      {'type': 'boolean'},
            'like_count':     {'type': 'integer'},
            'retweet_count':  {'type': 'integer'},
            # Kibana için timestamp alanları
            'created_at':     {'type': 'date'},
            'analyzed_at':    {'type': 'date', 'format': 'epoch_second'}
        }
    },
    'settings': {
        'number_of_shards':   3,   # Kafka partition sayısıyla eşleştir
        'number_of_replicas': 1,
        'refresh_interval':   '5s'  # Gerçek zamanlı görünürlük
    }
}

HASHTAG_MAPPING = {
    'mappings': {
        'properties': {
            'hashtag':           {'type': 'keyword'},
            'tweet_count':       {'type': 'integer'},
            'mean_compound':     {'type': 'float'},
            'engagement_score':  {'type': 'float'},
            'positive_pct':      {'type': 'float'},
            'negative_pct':      {'type': 'float'},
            'dominant_sentiment':{'type': 'keyword'},
            'window_start':      {'type': 'date', 'format': 'epoch_second'},
            'window_end':        {'type': 'date', 'format': 'epoch_second'},
        }
    }
}

class ElasticsearchSink:
    def __init__(self, hosts: list = ['http://localhost:9200']):
        self.es = Elasticsearch(hosts)
        self._ensure_indices()
        self.buffer = []
        self.BUFFER_SIZE = 100  # Her 100 belgede toplu yaz (bulk insert)

    def _ensure_indices(self):
        for idx, mapping in [('tweets', TWEET_MAPPING), ('hashtag_trends', HASHTAG_MAPPING)]:
            if not self.es.indices.exists(index=idx):
                self.es.indices.create(index=idx, body=mapping)
                logger.info(f'Elasticsearch index oluşturuldu: {idx}')

    def index_tweet(self, tweet: dict):
        self.buffer.append({
            '_index': 'tweets',
            '_id':     tweet.get('tweet_id'),
            '_source': tweet
        })
        if len(self.buffer) >= self.BUFFER_SIZE:
            self.flush()

    def index_aggregation(self, agg: dict):
        doc_id = f"{agg['hashtag']}_{agg['window_start']}"
        self.es.index(index='hashtag_trends', id=doc_id, body=agg)

    def flush(self):
        if self.buffer:
            helpers.bulk(self.es, self.buffer)
            logger.info(f'{len(self.buffer)} belge Elasticsearch\'e yazıldı.')
            self.buffer.clear()

def run_es_consumer(
    kafka_servers: str,
    result_topic:  str,
    aggreg_topic:  str
):
    """Kafka'dan Flink çıktısını okuyup Elasticsearch'e yazar."""
    tweet_consumer = KafkaConsumer(
        result_topic,
        bootstrap_servers=[kafka_servers],
        group_id='es_tweet_sink',
        value_deserializer=lambda x: json.loads(x.decode('utf-8')),
        auto_offset_reset='latest',
        max_poll_records=100
    )
    es_sink = ElasticsearchSink()

    logger.info('Elasticsearch Sink başlatıldı.')
    for msg in tweet_consumer:
        es_sink.index_tweet(msg.value)

if __name__ == '__main__':
    run_es_consumer('localhost:9092', 'sentiment_results', 'hashtag_aggregations')

# ================================================================
# Pipeline Performans İzleme ve Benchmarking
# ================================================================
import json, time, statistics
from kafka import KafkaConsumer
from collections import deque, defaultdict
from datetime import datetime

class PipelineMonitor:
    """
    Uçtan Uca Gecikme (End-to-End Latency) Ölçer.

    Gecikme = analyzed_at - ingestion_ts
    (Kafka'ya giriş anından Flink işleme bitimine kadar)
    """
    def __init__(self, window_size: int = 1000):
        self.latencies      = deque(maxlen=window_size)
        self.throughput_log = deque(maxlen=60)  # Son 60 saniye
        self.label_counts   = defaultdict(int)
        self.start_time     = time.time()
        self.total_messages = 0

    def record(self, tweet: dict):
        analyzed_at  = tweet.get('analyzed_at', time.time())
        ingestion_ts = tweet.get('ingestion_ts', analyzed_at)
        latency_ms   = (analyzed_at - ingestion_ts) * 1000

        if 0 < latency_ms < 60000:  # Makul aralık: 0-60sn
            self.latencies.append(latency_ms)

        label = tweet.get('final_label', 'UNKNOWN')
        self.label_counts[label] += 1
        self.total_messages += 1

    def report(self) -> dict:
        elapsed = time.time() - self.start_time
        if not self.latencies:
            return {}
        lat = list(self.latencies)
        return {
            'timestamp':         datetime.now().isoformat(),
            'total_messages':    self.total_messages,
            'throughput_msg_s':  round(self.total_messages / elapsed, 1),
            'latency_p50_ms':    round(statistics.median(lat), 1),
            'latency_p95_ms':    round(statistics.quantiles(lat, n=20)[18], 1),
            'latency_p99_ms':    round(statistics.quantiles(lat, n=100)[98], 1),
            'latency_max_ms':    round(max(lat), 1),
            'sentiment_dist':    dict(self.label_counts),
        }

def run_monitor(kafka_servers: str = 'localhost:9092',
               topic: str = 'sentiment_results'):
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=[kafka_servers],
        group_id='pipeline_monitor',
        value_deserializer=lambda x: json.loads(x.decode('utf-8')),
        auto_offset_reset='latest'
    )
    monitor = PipelineMonitor()
    print(f'Pipeline Monitor başlatıldı. Topic: {topic}')

    last_report = time.time()
    for msg in consumer:
        monitor.record(msg.value)

        # Her 10 saniyede bir rapor yazdır
        if time.time() - last_report > 10:
            report = monitor.report()
            print(json.dumps(report, indent=2, ensure_ascii=False))
            last_report = time.time()

if __name__ == '__main__':
    run_monitor()
